In [1]:
%load_ext autoreload
%autoreload 2

# Tabel abiverbid 
da + võima, saama ja tohtima
ma + pidama

**Ülesande püstitus**
    
Kõik abiverbid, kus esineb tabelis [list_da.csv](../lists/list_da.csv) olev verb ja verbil on otsene alluv deprel=xcomp, feats sisaldab inf

1. id; 2. sentence; 3. verb (st võima, saama, tohtima, pidama); 4. head (st verbi ülemuse lemma); 5. level (keeletase); 6. sub (nagu enne).



In [2]:
import pandas as pd
from datetime import datetime
from notebook_context import corpus_reader, LISTS_FOLDER

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

RESULTS_FILE= LISTS_FOLDER / f"results/aux_loend_dama_{date_time}.csv"
AUX_VERBS_LIST =  "./lists/104.list_aux.csv"

In [3]:
%%time

# verbid etteantud nimekirjast
df_verbs = pd.read_csv(AUX_VERBS_LIST)
my_verbs = list(df_verbs['lemma'].unique())

CPU times: user 2.14 ms, sys: 221 μs, total: 2.36 ms
Wall time: 4.17 ms


In [4]:
my_verbs

['võima', 'saama', 'tohtima', 'pidama']

In [5]:
%%time


collected_data = []
count = 0
for collection_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB")]
    aux_nodes = [v for v in graph.get_nodes_by_attributes(attrname="deprel", attrvalue="aux") if graph.nodes[v]["lemma"] in my_verbs]
    if not len(aux_nodes): 
        continue
    
    
    # aux on tegusõna 'võima', 'saama', 'tohtima', 'pidama'
    for aux in aux_nodes:
        aux_head = graph.nodes[aux]["head"]
        if aux_head not in verb_nodes:
            continue
       
        kids = [k for k in dpath[aux_head] if dpath[aux_head][k] == 1]
       
        #graph.draw_graph2(highlight=[aux_head, kids])
        d = {
            'id':  graph.get_metadata('sent_id'),
            'sentence':  graph.get_metadata('text'),
            'aux_head':  graph.nodes[aux_head]["lemma"],
            'aux':  graph.nodes[aux]["lemma"],
            'level':  graph.get_metadata('sent_level'),
            'sub': " ".join(
                        [graph.nodes[n]["form"] for n in sorted([aux_head] + kids)]
                    ),
            'sup': int( graph.nodes[aux_head]["feats"] is not None and "VerbForm" in graph.nodes[aux_head]["feats"] and graph.nodes[aux_head]["feats"]["VerbForm"] == 'Sup' ),
            'inf': int( graph.nodes[aux_head]["feats"] is not None and "VerbForm" in graph.nodes[aux_head]["feats"] and graph.nodes[aux_head]["feats"]["VerbForm"] == 'Inf' ),
            # 'aux_head_feats': graph.nodes[aux_head]["feats"],
            'keeletase': graph.get_metadata("doc").get("keeletase"),
            'emakeel': graph.get_metadata("doc").get("emakeel"),
            'klass': graph.get_metadata("doc").get("klass")
        }
            
        collected_data.append(d)

df = pd.DataFrame.from_dict(collected_data)
df.to_csv(RESULTS_FILE, index=None)
df.head(30)

../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
CPU times: user 12.2 s, sys: 74.2 ms, total: 12.3 s
Wall time: 12.3 s


,id,sentence,aux_head,aux,level,sub,sup,inf,keeletase,emakeel,klass
0,4094_1,Kitukas saab õelda selle lapse kohta kes räägi...,õtlema,saama,None,Kitukas saab õelda lapse .,0,1,0,1,3
1,4095_1,Kitukas saab õelda selle kohta kes kaebab kõik...,õtlema,saama,None,Kitukas saab õelda selle .,0,1,0,1,3
2,4097_1,Kitukas saab õelda lapse kohta kes räägib mida...,õtlema,saama,None,Kitukas saab õelda lapse .,0,1,0,1,3
3,4098_1,Kitukas saab õelda selle lapse kohta kes räägi...,õtlema,saama,None,Kitukas saab õelda lapse .,0,1,0,1,3
4,4098_3,"Minu ja kollide koolil on ainult 1 asi ühine, ...",andma,tohtima,None,nagu koolis ei tohi tappa anda peab,0,1,0,1,3
5,4099_1,Kitukas saab oelda(?) lapse kohta kes räägib k...,ütlema,saama,None,Kitukas saab oelda ( ? ) lapse .,0,1,0,1,3
6,4100_1,Sellise lapse koht saab öelda kitukas kes kittub.,ütlema,saama,None,koht saab öelda kitukas .,0,1,0,1,3
7,4103_4,Meil ei või mürada.,mürama,võima,None,Meil ei või mürada .,0,1,0,1,3
8,4105_2,"Selle pärast, et Keku luges neid Kollide kooli...",kaktlema,tohtima,None,tohib kakelda,0,1,0,1,3
9,4105_2,"Selle pärast, et Keku luges neid Kollide kooli...",kaktlema,tohtima,None,", kuid koolis ei tohi kakelda",0,1,0,1,3
